# Memory Experiment — 4D Geometric Code

High-rate codes from 4D lattices (arXiv:2506.15130).  
Key code: Hadamard [[96,6,8]].

In [ ]:
import sys
from pathlib import Path
import numpy as np

ROOT = Path("../..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from lightstim.ir.qec_system import QECSystem
from lightstim.protocols.memory import MemoryExperiment
from lightstim.qec_code.four_d_geo_code import FourDGeoCode, FourDGeoCodeExtractionBlock
from lightstim.noise.config import NoiseConfig


In [ ]:
p = 1e-3
noise_params = NoiseConfig(p_idle=p, p_1q=p, p_2q=p, p_meas=p, p_reset=p)


## 1. Det3 [[18,6,3]] — Smallest Code

In [ ]:
L_det3 = [[1,0,0,1],[0,1,0,1],[0,0,1,1],[0,0,0,3]]
code = FourDGeoCode(L=L_det3)
print(f"[[n={len(code.data_indices)}, k={code.num_logicals}, d={code.code_distance}]]")


In [ ]:
system = QECSystem()
system.add_patch(code, name="det3")

exp = MemoryExperiment(
    qec_system=system,
    extraction_block_class=FourDGeoCodeExtractionBlock,
    rounds=2,
    noise_params=noise_params,
    noise_model="circuit_level",
    basis="Z",
)
circuit = exp.build()
print(f"Qubits: {circuit.num_qubits}  Detectors: {circuit.num_detectors}")
circuit


In [ ]:
circuit.without_noise().diagram("detslice-with-ops-svg")


## 2. Hadamard [[96,6,8]] — Flagship Code

In [ ]:
L_hadamard = [[1,1,1,1],[0,2,0,2],[0,0,2,2],[0,0,0,4]]
code_h = FourDGeoCode(L=L_hadamard)
print(f"[[n={len(code_h.data_indices)}, k={code_h.num_logicals}, d={code_h.code_distance}]]")

system = QECSystem()
system.add_patch(code_h, name="hadamard")

exp = MemoryExperiment(
    qec_system=system,
    extraction_block_class=FourDGeoCodeExtractionBlock,
    rounds=2,
    noise_params=noise_params,
    noise_model="circuit_level",
    basis="Z",
)
circuit_h = exp.build()
print(f"Qubits: {circuit_h.num_qubits}  Detectors: {circuit_h.num_detectors}  k={code_h.num_logicals}")
# circuit_h.without_noise().diagram("detslice-with-ops-svg")  # large — uncomment to visualize


## 3. All Lattices from Paper (Table 5)

In [ ]:
lattices = {
    "Det3 [[18,6,3]]":     [[1,0,0,1],[0,1,0,1],[0,0,1,1],[0,0,0,3]],
    "Hadamard [[96,6,8]]": [[1,1,1,1],[0,2,0,2],[0,0,2,2],[0,0,0,4]],
}

for name, L in lattices.items():
    code = FourDGeoCode(L=L)
    system = QECSystem()
    system.add_patch(code, name="code")
    exp = MemoryExperiment(
        qec_system=system,
        extraction_block_class=FourDGeoCodeExtractionBlock,
        rounds=2,
        noise_params=noise_params,
        noise_model="circuit_level",
        basis="Z",
    )
    circ = exp.build()
    n = len(code.data_indices)
    k = code.num_logicals
    print(f"{name}: n={n}, k={k}, {circ.num_qubits} total qubits, {circ.num_detectors} detectors")
